# Relay — Self-Hosted Qwen3-Coder (vLLM) on Kaggle

This notebook provides an automated runbook to deploy **Qwen3-Coder-30B-A3B-Instruct** on Kaggle (2× NVIDIA Tesla T4) using **vLLM 0.29.0** and connect it to your local **Relay** gateway via an encrypted **Cloudflare Quick Tunnel**.

### System Architecture
```text
Local Relay Gateway  ──►  Cloudflare Quick Tunnel  ──►  vLLM Engine (TP=2)  ──►  Qwen3-Coder-30B
 (developer machine)     (*.trycloudflare.com)       (port 8000 on dual T4)     (30.5B MoE, AWQ)
```

### Verified Target Environment
- **Hardware**: 2× NVIDIA Tesla T4 (16 GB VRAM each, ~15,109 MiB usable).
- **Model**: `QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ` (fits in ~7.6 GB VRAM per GPU).
- **Served Model Name**: `qwen3-coder-30b` (single clean alias).
- **vLLM Configuration**: `vllm serve --tensor-parallel-size 2 --dtype float16 --quantization awq --max-model-len 4096 --max-num-seqs 4 --gpu-memory-utilization 0.85 --enforce-eager --trust-remote-code`

> [!NOTE]
> **Ephemeral Environment**: Kaggle GPU sessions terminate after 9–12 hours. This setup is intended for development, integration testing, and evaluation of Relay's multi-provider routing.

### How this Runbook Works
This notebook **orchestrates the version-controlled management scripts** from the Relay repository (`infra/kaggle/`):
- `qwen-vllm.sh`: Background vLLM lifecycle, process isolation, local readiness, and cleanup.
- `cloudflared.sh`: Tunnel binary provisioning, background daemon, and dynamic URL extraction.
- `diagnostics.sh`: 10-point health check across GPU, CUDA, port, processes, and network.


---
## Step 1: Environment Check
Inspect the container OS, Python runtime, working directory, and available system RAM.


In [ ]:
import os
import sys
import platform

print(f"Python:    {sys.version.split()[0]} ({sys.executable})")
print(f"Platform:  {platform.platform()}")
print(f"Directory: {os.getcwd()}")

try:
    with open("/proc/meminfo", "r") as f:
        for line in f:
            if any(k in line for k in ("MemTotal", "MemAvailable")):
                print(f"Memory:    {line.strip()}")
except Exception as e:
    print(f"Memory info unavailable: {e}")


---
## Step 2: GPU & VRAM Verification
Confirm that two NVIDIA Tesla T4 GPUs are allocated with clean memory.

> [!WARNING]
> If fewer than 2 GPUs are detected, open the right sidebar -> **Notebook settings** -> **Accelerator** -> **GPU T4 x2**.


In [ ]:
import subprocess
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("nvidia-smi not found. Select 'GPU T4 x2' in Kaggle Notebook Settings.")

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv"], check=False)

try:
    import torch
    dev_count = torch.cuda.device_count()
    print(f"\nPyTorch CUDA Devices: {dev_count}")
    for i in range(dev_count):
        free_mb = torch.cuda.mem_get_info(i)[0] / (1024 ** 2)
        total_mb = torch.cuda.mem_get_info(i)[1] / (1024 ** 2)
        print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): {free_mb:.0f} MiB free / {total_mb:.0f} MiB total")
    
    if dev_count < 2:
        print("\n[FAIL] Tensor parallelism requires 2 GPUs. Change accelerator to 'GPU T4 x2'.")
    else:
        print("\n[PASS] Dual GPU environment ready.")
except Exception as e:
    print(f"PyTorch CUDA check error: {e}")


---
## Step 3: Clone Relay Repository
Clone the repository into `/kaggle/working/Relay` to obtain the infrastructure scripts (`infra/kaggle/`), then switch the working directory.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/mosabbir-maruf/Relay.git"
CLONE_DEST = "/kaggle/working/Relay"

if not os.path.exists(CLONE_DEST):
    print(f"Cloning {REPO_URL} -> {CLONE_DEST}...")
    subprocess.run(["git", "clone", REPO_URL, CLONE_DEST], check=True)
else:
    print(f"Repository already present at {CLONE_DEST}.")

os.chdir(CLONE_DEST)
print(f"Working directory: {os.getcwd()}")

commit_sha = subprocess.check_output(
    ["git", "log", "-1", "--pretty=format:%h - %s (%ci)"], text=True
).strip()
print(f"Active commit:     {commit_sha}")


---
## Step 4: Verify Infrastructure Scripts
Confirm the management scripts exist in `infra/kaggle/`.


In [ ]:
import os

scripts = [
    "infra/kaggle/README.md",
    "infra/kaggle/qwen-vllm.sh",
    "infra/kaggle/cloudflared.sh",
    "infra/kaggle/diagnostics.sh"
]

all_ok = True
for path in scripts:
    exists = os.path.isfile(path)
    size = os.path.getsize(path) if exists else 0
    status = "OK" if exists and size > 0 else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"[{status:7s}] {path} ({size:,} bytes)")

assert all_ok, "Required infrastructure scripts are missing from repository clone!"


---
## Step 5: Apply Execute Permissions (`chmod +x`)
Kaggle's container filesystem requires explicit POSIX execute permissions (`+x`) before scripts can be executed directly.


In [ ]:
import os
import subprocess
import glob

for script in sorted(glob.glob("infra/kaggle/*.sh")):
    subprocess.run(["chmod", "+x", script], check=True)
    print(f"chmod +x {script} -> Executable: {os.access(script, os.X_OK)}")


---
## Step 6: vLLM Preflight Check
Validates GPUs, verifies or installs the `vllm` CLI package, and checks port 8000 availability.


In [ ]:
import subprocess
import shutil

if not shutil.which("vllm"):
    print("vLLM CLI not found. Installing vllm package (~60s)...")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "vllm"], check=True)

subprocess.run(["./infra/kaggle/qwen-vllm.sh", "check"], check=True)


---
## Step 7: System Diagnostics
Runs the 10-point diagnostic suite to check GPU memory, PyTorch CUDA allocation, and port 8000 before launching services.


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/diagnostics.sh", "all"], check=False)


---
## Step 8: Safe Process Cleanup
Safely terminates any stale vLLM processes bound to port 8000 and frees leaked GPU VRAM without affecting Jupyter.


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/qwen-vllm.sh", "clean"], check=True)


---
## Step 9: Launch vLLM in Background
Launches `vllm serve QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ` in the background (TP=2, FP16, AWQ, eager execution). Logs stream to `/kaggle/working/vllm_server.log`.

*Remediation if startup fails*: Run `./infra/kaggle/qwen-vllm.sh logs 50` to inspect error output.


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/qwen-vllm.sh", "start"], check=True)


---
## Step 10: Wait for vLLM Readiness
Polls `http://127.0.0.1:8000/v1/models` until weights (~15.2 GB) are downloaded and the KV cache compiles (typically 2–4 minutes).


In [ ]:
import time
import urllib.request
import json
import subprocess

HEALTH_URL = "http://127.0.0.1:8000/v1/models"
TIMEOUT_SECS = 600
POLL_INTERVAL = 5

print(f"Waiting for vLLM readiness at {HEALTH_URL} (timeout: {TIMEOUT_SECS}s)...")
start_time = time.time()
ready = False

while time.time() - start_time < TIMEOUT_SECS:
    elapsed = int(time.time() - start_time)
    try:
        req = urllib.request.Request(HEALTH_URL)
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode("utf-8"))
                models = [m.get("id") for m in data.get("data", [])]
                print(f"\n[READY] vLLM is online after {elapsed}s!")
                print(f"Available models: {models}")
                ready = True
                break
    except Exception:
        if elapsed % 20 == 0:
            print(f"[WAITING] Loading weights and compiling KV cache... ({elapsed}s elapsed)")
    time.sleep(POLL_INTERVAL)

if not ready:
    print("\n[TIMEOUT] vLLM did not become ready in time. Last 30 log lines:")
    subprocess.run(["./infra/kaggle/qwen-vllm.sh", "logs", "30"], check=False)
    raise TimeoutError("vLLM readiness timeout. Inspect log output above.")


---
## Step 11: Local Model Check
Verifies that `/v1/models` returns the exact single served model name: `qwen3-coder-30b`.


In [ ]:
import urllib.request
import json

req = urllib.request.Request("http://127.0.0.1:8000/v1/models")
with urllib.request.urlopen(req, timeout=5) as resp:
    data = json.loads(resp.read().decode("utf-8"))

models = [m["id"] for m in data.get("data", [])]
print(f"Models returned: {models}")
assert "qwen3-coder-30b" in models, f"Expected 'qwen3-coder-30b' in {models}"
print("[PASS] Model alias 'qwen3-coder-30b' confirmed.")


---
## Step 12: Local Inference Test
Executes a smoke test completion (`"Reply with only the word PONG"`) via `./infra/kaggle/qwen-vllm.sh test`.


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/qwen-vllm.sh", "test"], check=True)


---
## Step 13: Local Streaming Test (SSE)
Verifies Server-Sent Events (SSE) token streaming (`stream: true`) and validates the terminal `data: [DONE]` frame.


In [ ]:
import urllib.request
import json

payload = {
    "model": "qwen3-coder-30b",
    "messages": [{"role": "user", "content": "Count from 1 to 5"}],
    "temperature": 0.0,
    "max_tokens": 30,
    "stream": True
}

req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

received_done = False
with urllib.request.urlopen(req, timeout=30) as resp:
    for raw in resp:
        line = raw.decode("utf-8").strip()
        if line == "data: [DONE]":
            received_done = True
            break
        if line.startswith("data: "):
            chunk = json.loads(line[6:])
            delta = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
            print(delta, end="", flush=True)

print(f"\n\n[PASS] Stream finished with [DONE]: {received_done}")
assert received_done, "Streaming did not conclude with data: [DONE]"


---
## Step 14: cloudflared Binary Verification
Confirms `cloudflared` is available at `/kaggle/working/cloudflared` (auto-downloads official binary if missing).


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/cloudflared.sh", "check"], check=True)


---
## Step 15: Start Cloudflare Quick Tunnel
Launches `cloudflared` in the background targeting `http://127.0.0.1:8000` and dynamically extracts the assigned public URL.

*Remediation if URL not detected*: Verify Internet is enabled in Kaggle settings, or inspect `/kaggle/working/cloudflared.log`.


In [ ]:
import subprocess
import time

subprocess.run(["./infra/kaggle/cloudflared.sh", "start"], check=True)

PUBLIC_TUNNEL_URL = ""
for _ in range(15):
    try:
        url = subprocess.check_output(["./infra/kaggle/cloudflared.sh", "url"], text=True).strip()
        if "trycloudflare.com" in url:
            PUBLIC_TUNNEL_URL = url
            break
    except Exception:
        pass
    time.sleep(2)

if PUBLIC_TUNNEL_URL:
    print(f"\nSUCCESS: Public Tunnel URL -> {PUBLIC_TUNNEL_URL}")
else:
    print("\n[ERROR] Could not extract tunnel URL. Tunnel logs:")
    subprocess.run(["./infra/kaggle/cloudflared.sh", "logs", "20"], check=False)
    raise RuntimeError("Failed to obtain Cloudflare Quick Tunnel URL.")


---
## Step 16: Public Model Discovery Test
Tests ingress through Cloudflare edge to the vLLM server: `<PUBLIC_TUNNEL_URL>/v1/models`.


In [ ]:
import urllib.request
import json

url = f"{PUBLIC_TUNNEL_URL}/v1/models"
req = urllib.request.Request(url)
with urllib.request.urlopen(req, timeout=15) as resp:
    data = json.loads(resp.read().decode("utf-8"))

models = [m["id"] for m in data.get("data", [])]
print(f"HTTP Status: {resp.status}")
print(f"Public models: {models}")
assert "qwen3-coder-30b" in models, f"Model 'qwen3-coder-30b' not found in {models}"
print("[PASS] Public model discovery verified.")


---
## Step 17: Public End-to-End Inference Test
Sends a test chat completion request through the public HTTPS endpoint to verify full external connectivity:
`Client -> Cloudflare Edge -> Kaggle -> vLLM -> Qwen3-Coder`.


In [ ]:
import urllib.request
import json

payload = {
    "model": "qwen3-coder-30b",
    "messages": [{"role": "user", "content": "Reply with only the word PONG"}],
    "temperature": 0.0,
    "max_tokens": 16
}

req = urllib.request.Request(
    f"{PUBLIC_TUNNEL_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=30) as resp:
    res = json.loads(resp.read().decode("utf-8"))

answer = res["choices"][0]["message"]["content"].strip()
print(f"HTTP Status: {resp.status}")
print(f"Response:    {answer}")
assert "PONG" in answer.upper(), f"Expected PONG, got: {answer}"
print("[PASS] End-to-end public inference verified!")


---
## Step 18: Relay Gateway Configuration
Paste this configuration snippet into your local `.env` file on your development machine.

> [!IMPORTANT]
> Quick Tunnel URLs change every time `cloudflared` starts. Do not commit temporary URLs to git.


In [ ]:
print("=" * 60)
print("Add this to your local Relay .env file:")
print("=" * 60)
print(f"QWEN_BASE_URL={PUBLIC_TUNNEL_URL}/v1")
print(f"QWEN_MODEL=qwen3-coder-30b")
print("=" * 60)
print("\nTo run Relay locally:")
print("  pnpm dev")
print("\nTo test routing through Relay:")
print("  curl -X POST http://localhost:3000/v1/chat/completions \\")
print('    -H "Content-Type: application/json" \\')
payload_example = '{"model": "qwen3-coder-30b", "messages": [{"role": "user", "content": "Hello!"}]}'
print(f"    -d '{payload_example}'")


---
## Step 19: Operational Status Check
Inspect daemon processes, active public URL, and current GPU memory allocation.


In [ ]:
import subprocess

print("--- vLLM Server Status ---")
subprocess.run(["./infra/kaggle/qwen-vllm.sh", "status"], check=False)

print("\n--- Cloudflare Tunnel Status ---")
subprocess.run(["./infra/kaggle/cloudflared.sh", "status"], check=False)

print("\n--- GPU Memory Allocation ---")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"], check=False)


---
## Step 20: Safe Shutdown
Terminate the tunnel and vLLM processes cleanly and confirm that all GPU memory is freed back to Kaggle.


In [ ]:
import subprocess
import time

print("Stopping Cloudflare Tunnel...")
subprocess.run(["./infra/kaggle/cloudflared.sh", "stop"], check=False)

print("\nStopping vLLM Server...")
subprocess.run(["./infra/kaggle/qwen-vllm.sh", "stop"], check=False)

print("\nWaiting for GPU memory release...")
time.sleep(3)
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free", "--format=csv"], check=False)

print("\n[SUCCESS] All services stopped cleanly. You may now close the Kaggle session.")


---
## Troubleshooting Reference

| Symptom / Error | Root Cause | Actionable Remediation |
| :--- | :--- | :--- |
| **GPU count < 2** | Kaggle accelerator set to CPU or single GPU | Right sidebar -> **Notebook settings** -> **Accelerator** -> **GPU T4 x2**. |
| **`vllm: error: unrecognized subcommand 'server'`** | Deprecated command name | Use canonical `vllm serve`. |
| **`unrecognized arguments: --swap-space`** | Deprecated argument in vLLM 0.29.0 | Omit `--swap-space`. Manage headroom with `--gpu-memory-utilization 0.85`. |
| **Compound model name (`model,alias`)** | Comma-separated alias string | Use single string: `--served-model-name qwen3-coder-30b`. |
| **Stale workers / High VRAM on startup** | Prior run terminated abruptly | Run `./infra/kaggle/qwen-vllm.sh clean` and verify VRAM dropped to 0 MiB before restart. |
| **Port 8000 occupied** | Previous server instance still listening | Run `./infra/kaggle/qwen-vllm.sh clean`. |
| **Readiness timeout** | Weights (~15 GB) still downloading or KV cache compiling | Run `tail -n 50 /kaggle/working/vllm_server.log` or `./infra/kaggle/qwen-vllm.sh logs 50`. Initial load takes 2–4 min. |
| **`cloudflared` binary missing** | Binary not yet downloaded | Run `./infra/kaggle/cloudflared.sh check` to download official release. |
| **Tunnel URL not detected** | Network delay or internet disabled in Kaggle | Check `/kaggle/working/cloudflared.log`. Verify **Internet: On** in Kaggle settings. |
| **Public endpoint fails (HTTP 530 / Error 1033)** | Break in connectivity chain | Follow the 6-step isolation order below. |

### 6-Step Failure Isolation Hierarchy
When requests through the public URL fail, diagnose in this exact order:
1. `curl http://127.0.0.1:8000/v1/models` ──► If fails: vLLM server crashed or loading. Check `vllm_server.log`.
2. `./infra/kaggle/qwen-vllm.sh test` ──► If fails: Forward pass OOM or engine failure.
3. `./infra/kaggle/cloudflared.sh status` ──► If fails: `cloudflared` process stopped.
4. `./infra/kaggle/cloudflared.sh logs` ──► If errors: Network disconnect or edge throttle.
5. `curl https://<subdomain>.trycloudflare.com/v1/models` ──► If fails: Cloudflare edge DNS propagation.
6. `curl -X POST https://<subdomain>.trycloudflare.com/v1/chat/completions ...` ──► If fails: Request timeout or payload error.
